# 문서 이미지 자동 교정 및 향상 시스템
# 文档图像自动矫正与增强系统
# Document Image Auto-Correction & Enhancement

**Google Colab Edition**

스마트폰으로 촬영된 문서 이미지의 기하학적 왜곡을 보정하고 텍스트 가독성을 향상시키는 시스템입니다.
本系统用于矫正手机拍摄文档图像的几何失真并提高文本可读性。

- ⚠️ Machine Learning / Deep Learning 사용 금지 / 禁止使用机器学习/深度学习
- ✅ 전통적 Computer Vision + Image Processing 기법만 사용 / 仅使用传统计算机视觉与图像处理技术 (OpenCV, NumPy)

## Pipeline (11 Steps / 11단계 / 11步骤)

```
Input → Grayscale → Gaussian Blur → Canny Edge → Edge Dilation
      → Contour Detection → Convex Hull → Quad Approximation
      → Corner Ordering → Perspective Transform
      → CLAHE → Adaptive Threshold → Morphology → Output
```

In [ ]:
# 1. 환경 설정 및 라이브러리 import
# 1. 环境设置与库导入
# Google Colab에는 OpenCV, NumPy, Matplotlib가 기본 설치되어 있습니다.
# Google Colab 已预装 OpenCV、NumPy、Matplotlib。
# 만약 누락된 패키지가 있다면 아래 주석을 해제하세요 / 如有缺失请取消下面注释:
# !pip install opencv-python numpy matplotlib

import cv2
import numpy as np
import os
import time
import glob
from pathlib import Path
import matplotlib.pyplot as plt
from google.colab import files
from google.colab.patches import cv2_imshow
from IPython.display import display, Image as IPImage, HTML
import ipywidgets as widgets
from IPython.display import clear_output

print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version: {np.__version__}")
print("All libraries loaded successfully! / 모든 라이브러리 로드 완료! / 所有库加载成功！")

---
## 2. Utility Functions / 유틸리티 함수 / 工具函数 (Image I/O)

In [ ]:
# 2. 이미지 입출력 유틸리티 / 图像输入输出工具

def load_image(path):
    """Load image from file path as BGR ndarray.
    이미지 파일을 BGR ndarray로 로드합니다.
    从文件路径加载图像为BGR ndarray。
    Uses: cv2.imread()"""
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Image file not found / 파일 없음 / 文件不存在: {path}")
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Failed to read image / 이미지 읽기 실패 / 图像读取失败: {path}")
    return img


def save_image(img, path):
    """Save image to disk.
    이미지를 디스크에 저장합니다.
    将图像保存到磁盘。
    Uses: cv2.imwrite()"""
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    cv2.imwrite(path, img)


def save_step(img, output_dir, step_num, step_name):
    """Save intermediate result as '{step_num:02d}_{step_name}.jpg'.
    중간 결과를 '{step_num:02d}_{step_name}.jpg' 형식으로 저장합니다.
    将中间结果保存为 '{step_num:02d}_{step_name}.jpg' 格式。
    Uses: cv2.imwrite()"""
    filename = f"{step_num:02d}_{step_name}.jpg"
    filepath = os.path.join(output_dir, filename)
    os.makedirs(output_dir, exist_ok=True)
    cv2.imwrite(filepath, img)


def show_images(images, titles, cols=3, figsize=(18, 10)):
    """Display multiple images in a grid using matplotlib.
    여러 이미지를 matplotlib 그리드로 표시합니다.
    使用matplotlib以网格形式显示多张图像。"""
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.flatten() if rows > 1 or cols > 1 else [axes]
    for i, (img, title) in enumerate(zip(images, titles)):
        if len(img.shape) == 2:
            axes[i].imshow(img, cmap='gray')
        else:
            axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[i].set_title(title, fontsize=10)
        axes[i].axis('off')
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
    plt.tight_layout()
    plt.show()

print("Utility functions defined. / 유틸리티 함수 정의 완료. / 工具函数已定义。")

---
## 3. Preprocessing Module / 전처리 모듈 / 预处理模块
Step 2-3: Grayscale Conversion + Gaussian Filtering

In [ ]:
# 3. 전처리 모듈 / 预处理模块

def to_grayscale(img):
    """Convert BGR to grayscale.
    BGR 이미지를 그레이스케일로 변환합니다.
    将BGR图像转换为灰度图。
    Formula / 공식 / 公式: Gray = 0.299R + 0.587G + 0.114B
    Uses: cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)"""
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)


def apply_gaussian_blur(gray, kernel_size=(5, 5), sigma=0):
    """Apply Gaussian filter to suppress high-frequency noise.
    가우시안 필터를 적용하여 고주파 노이즈를 제거합니다.
    应用高斯滤波以抑制高频噪声。
    Uses: cv2.GaussianBlur(gray, kernel_size, sigma)"""
    return cv2.GaussianBlur(gray, kernel_size, sigma)

print("Preprocessing functions defined. / 전처리 함수 정의 완료. / 预处理函数已定义。")

---
## 4. Detection Module / 문서 검출 모듈 / 文档检测模块
Step 4-6: Canny Edge → Edge Dilation → Contour Detection → Quadrilateral Approximation

In [ ]:
# 4. 문서 검출 모듈 / 文档检测模块

def detect_edges(blurred, low=75, high=200):
    """Extract edges from blurred grayscale image.
    블러 처리된 그레이스케일 이미지에서 에지를 추출합니다.
    从模糊灰度图像中提取边缘。
    Uses: cv2.Canny(blurred, low, high)"""
    return cv2.Canny(blurred, low, high)


def dilate_edges(edges, kernel_size=(3, 3), iterations=2):
    """Dilate edge map to connect fragmented edge segments.
    에지 맵을 팽창하여 조각난 에지 세그먼트를 연결합니다.
    膨胀边缘图以连接碎片化的边缘段。
    Uses: cv2.dilate()"""
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, kernel_size)
    return cv2.dilate(edges, kernel, iterations=iterations)


def find_contours(edges):
    """Extract contours sorted by area descending (top 10).
    면적 기준 내림차순으로 정렬된 윤곽선을 추출합니다 (상위 10개).
    提取轮廓并按面积降序排序（前10个）。
    Uses: cv2.findContours(edges, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)"""
    contours, _ = cv2.findContours(edges, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)
    return contours[:10]


def _approx_quad(contour):
    """Try polygon approximation at various epsilon ratios.
    다양한 epsilon 비율로 다각형 근사를 시도합니다.
    以不同的epsilon比例尝试多边形逼近。
    Uses: cv2.approxPolyDP()"""
    peri = cv2.arcLength(contour, True)
    for ratio in (0.02, 0.03, 0.05, 0.08, 0.10):
        epsilon = ratio * peri
        approx = cv2.approxPolyDP(contour, epsilon, True)
        if len(approx) == 4:
            return approx
    return None


def find_document_contour(contours, img_area=0):
    """Find the best quadrilateral document candidate.
    최적의 사각형 문서 후보를 찾습니다.
    寻找最佳四边形文档候选区域。
    Strategy / 전략 / 策略:
      1. Direct polygon approximation per contour / 각 윤곽선 직접 다각형 근사 / 对每个轮廓直接进行多边形逼近
      2. Convex hull + approximation for partial edges / 부분 에지에 대한 볼록 껍질 + 근사 / 凸包+逼近处理部分边缘
      3. Combined contour convex hull as fallback / 전체 윤곽선 결합 볼록 껍질 폴백 / 组合所有轮廓的凸包作为后备方案
    Uses: cv2.approxPolyDP(), cv2.convexHull(), cv2.contourArea()
    """
    min_area = img_area * 0.02 if img_area > 0 else 5000

    for contour in contours:
        area = cv2.contourArea(contour)
        if area < min_area:
            continue
        quad = _approx_quad(contour)
        if quad is not None:
            return quad
        hull = cv2.convexHull(contour)
        quad = _approx_quad(hull)
        if quad is not None:
            return quad

    if len(contours) >= 2:
        combined = np.vstack(contours)
        hull = cv2.convexHull(combined)
        hull_area = cv2.contourArea(hull)
        if hull_area >= min_area:
            quad = _approx_quad(hull)
            if quad is not None:
                return quad
    return None


def draw_contour(img, contour):
    """Draw the detected document boundary in green.
    검출된 문서 경계를 녹색으로 그립니다.
    以绿色绘制检测到的文档边界。
    Uses: cv2.drawContours()"""
    result = img.copy()
    cv2.drawContours(result, [contour], -1, (0, 255, 0), 3)
    return result

print("Detection functions defined. / 검출 함수 정의 완료. / 检测函数已定义。")

---
## 5. Geometry Module / 기하학적 보정 모듈 / 几何校正模块
Step 7-8: Corner Ordering + Perspective Transformation

In [ ]:
# 5. 기하학적 보정 모듈 / 几何校正模块

def _dist(p1, p2):
    """Euclidean distance between two points.
    두 점 사이의 유클리드 거리.
    两点之间的欧氏距离。"""
    return np.sqrt((p2[0] - p1[0]) ** 2 + (p2[1] - p1[1]) ** 2)


def order_points(pts):
    """Order 4 corner points as [TL, TR, BR, BL].
    4개 꼭짓점을 [TL, TR, BR, BL] 순서로 정렬합니다.
    将4个角点排序为 [左上, 右上, 右下, 左下]。
    TL: min x+y,  BR: max x+y,  TR: max x-y,  BL: min x-y.
    Returns (4,2) float32 ndarray."""
    pts = pts.reshape(4, 2)
    ordered = np.zeros((4, 2), dtype=np.float32)
    sums = pts.sum(axis=1)
    diffs = np.diff(pts, axis=1)
    ordered[0] = pts[np.argmin(sums)]   # TL / 좌상단 / 左上
    ordered[2] = pts[np.argmax(sums)]   # BR / 우하단 / 右下
    ordered[1] = pts[np.argmax(diffs)]  # TR / 우상단 / 右上
    ordered[3] = pts[np.argmin(diffs)]  # BL / 좌하단 / 左下
    return ordered


def calculate_output_size(ordered_pts):
    """Compute output width and height from ordered corner points.
    정렬된 꼭짓점으로부터 출력 너비와 높이를 계산합니다.
    从排序后的角点计算输出宽度和高度。
    width  = max(dist(BL,BR), dist(TL,TR))
    height = max(dist(TR,BR), dist(TL,BL))"""
    tl, tr, br, bl = ordered_pts
    width = int(max(_dist(bl, br), _dist(tl, tr)))
    height = int(max(_dist(tr, br), _dist(tl, bl)))
    return width, height


def apply_perspective_transform(img, ordered_pts):
    """Apply perspective correction to straighten the document.
    투시 변환을 적용하여 문서를 반듯하게 보정합니다.
    应用透视变换将文档拉直。
    Uses: cv2.getPerspectiveTransform(), cv2.warpPerspective()"""
    width, height = calculate_output_size(ordered_pts)
    dst_pts = np.array([
        [0, 0],
        [width - 1, 0],
        [width - 1, height - 1],
        [0, height - 1],
    ], dtype=np.float32)
    M = cv2.getPerspectiveTransform(ordered_pts, dst_pts)
    warped = cv2.warpPerspective(img, M, (width, height))
    return warped

print("Geometry functions defined. / 기하 함수 정의 완료. / 几何函数已定义。")

---
## 6. Enhancement Module / 이미지 향상 모듈 / 图像增强模块
Step 9-10: CLAHE + Adaptive Thresholding + Morphological Processing

In [ ]:
# 6. 이미지 향상 모듈 / 图像增强模块

def apply_clahe(img_gray, clip_limit=2.0, tile_size=(8, 8)):
    """Enhance local contrast using CLAHE.
    CLAHE를 사용하여 지역 대비를 향상시킵니다.
    使用CLAHE增强局部对比度。
    Uses: cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_size)"""
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_size)
    return clahe.apply(img_gray)


def apply_adaptive_threshold(img_gray, block_size=11, C=2):
    """Binarize image with adaptive thresholding to handle uneven illumination.
    적응형 임계값 처리로 불균일한 조명을 보정하며 이미지를 이진화합니다.
    使用自适应阈值处理来应对不均匀光照，对图像进行二值化。
    Uses: cv2.adaptiveThreshold()"""
    return cv2.adaptiveThreshold(
        img_gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        block_size, C,
    )


def apply_morphology(img_binary, kernel_size=(3, 3)):
    """Clean binary image with morphological closing.
    형태학적 닫힘 연산으로 이진 이미지의 잡음을 정리합니다.
    使用形态学闭运算清理二值图像噪声。
    Uses: cv2.morphologyEx(MORPH_CLOSE)"""
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, kernel_size)
    return cv2.morphologyEx(img_binary, cv2.MORPH_CLOSE, kernel)


def enhance(warped):
    """Run full enhancement pipeline on the perspective-corrected image.
    투시 보정된 이미지에 전체 향상 파이프라인을 적용합니다.
    对透视矫正后的图像执行完整增强流水线。
    Steps: grayscale → CLAHE → adaptive threshold → morphology.
    Returns BGR image."""
    gray = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)
    clahed = apply_clahe(gray)
    thresh = apply_adaptive_threshold(clahed)
    cleaned = apply_morphology(thresh)
    return cv2.cvtColor(cleaned, cv2.COLOR_GRAY2BGR)

print("Enhancement functions defined. / 향상 함수 정의 완료. / 增强函数已定义。")

---
## 7. Full Pipeline / 전체 파이프라인 / 完整流水线

In [ ]:
# 7. 전체 파이프라인 / 完整流水线

def run_pipeline(image, output_dir="output", save_steps=True, basename="document"):
    """Run the full 11-step document correction pipeline.
    전체 11단계 문서 교정 파이프라인을 실행합니다.
    运行完整的11步文档矫正流水线。

    Args / 인자 / 参数:
        image: BGR image (numpy array) or path string.
               BGR 이미지(numpy 배열) 또는 파일 경로 문자열.
               BGR图像(numpy数组)或文件路径字符串。
        output_dir: Directory for output images / 출력 디렉터리 / 输出目录.
        save_steps: Save intermediate step images / 중간 단계 이미지 저장 여부 / 是否保存中间步骤图像.
        basename: Base name for output files / 출력 파일 기본 이름 / 输出文件基础名称.

    Returns / 반환값 / 返回值:
        dict: {success, output_path, processing_time, steps_images, warped, result}
    """
    start_time = time.time()
    output_path = os.path.join(output_dir, f"{basename}_corrected.jpg")
    steps_dir = os.path.join(output_dir, f"{basename}_steps") if save_steps else None
    steps_images = {}

    # Load image if path provided / 경로면 로드, 배열이면 복사 / 如果是路径则加载，否则复制
    if isinstance(image, str):
        original = load_image(image)
    else:
        original = image.copy()

    if save_steps:
        save_step(original, steps_dir, 1, "original")
        steps_images['01_original'] = original.copy()

    # Step 2: Grayscale / 그레이스케일 / 灰度化
    gray = to_grayscale(original)
    if save_steps:
        save_step(gray, steps_dir, 2, "grayscale")
        steps_images['02_grayscale'] = gray.copy()

    # Step 3: Gaussian blur / 가우시안 블러 / 高斯模糊
    blurred = apply_gaussian_blur(gray)
    if save_steps:
        save_step(blurred, steps_dir, 3, "blurred")
        steps_images['03_blurred'] = blurred.copy()

    # Step 4: Canny edge detection / 케니 에지 검출 / Canny边缘检测
    edges = detect_edges(blurred)
    if save_steps:
        save_step(edges, steps_dir, 4, "edges")
        steps_images['04_edges'] = edges.copy()

    # Step 4b: Dilate edges to connect fragments / 에지 팽창으로 조각 연결 / 膨胀边缘连接碎片
    dilated = dilate_edges(edges)

    # Step 5-6: Contour detection + quadrilateral approximation / 윤곽선 검출 + 사각형 근사 / 轮廓检测+四边形逼近
    contours = find_contours(dilated)
    img_area = original.shape[0] * original.shape[1]
    doc_contour = find_document_contour(contours, img_area)

    if save_steps and contours:
        display_contour = doc_contour if doc_contour is not None else contours[0]
        contour_img = draw_contour(original, display_contour)
        save_step(contour_img, steps_dir, 5, "contours")
        steps_images['05_contours'] = contour_img.copy()

    if doc_contour is not None:
        # Step 7-8: Corner ordering + perspective transform / 꼭짓점 정렬 + 투시 변환 / 角点排序+透视变换
        ordered_pts = order_points(doc_contour)
        warped = apply_perspective_transform(original, ordered_pts)
        if save_steps:
            save_step(warped, steps_dir, 6, "perspective")
            steps_images['06_perspective'] = warped.copy()

        # Step 9-10: Enhancement / 화질 향상 / 图像增强
        result = enhance(warped)
        if save_steps:
            save_step(result, steps_dir, 7, "enhanced")
            steps_images['07_enhanced'] = result.copy()

        success = True
    else:
        # Detection failed — fallback to enhancement on original
        # 검출 실패 — 원본에 향상 처리만 적용 / 检测失败 — 仅对原图进行增强
        result = enhance(original)
        if save_steps:
            save_step(original, steps_dir, 6, "perspective")
            save_step(result, steps_dir, 7, "enhanced")
            steps_images['06_perspective'] = original.copy()
            steps_images['07_enhanced'] = result.copy()
        warped = original
        success = False

    save_image(result, output_path)
    elapsed = time.time() - start_time

    return {
        "success": success,
        "output_path": output_path,
        "processing_time": elapsed,
        "steps_images": steps_images,
        "warped": warped,
        "result": result,
        "original": original,
    }

print("Pipeline function defined. / 파이프라인 함수 정의 완료. / 流水线函数已定义。")

---
## Evaluation Metrics / 평가지표 / 评估指标

처리 품질을 정량적으로 평가하는 지표들입니다.
用于定量评估处理质量的指标。

- **Laplacian Variance** — 이미지 선명도 점수 (값이 높을수록 선명) / 图像清晰度评分 (值越高越清晰)
- **SSIM** — 처리 전후 구조적 유사도 (0~1, 1에 가까울수록 원본과 유사) / 处理前后结构相似度 (0~1, 越接近1越相似)


In [ ]:
# 평가 지표 모듈 / 评估指标模块
# Evaluation Metrics Module
#
# Google Colab에는 scikit-image가 기본 설치되어 있습니다.
# Google Colab 已预装 scikit-image。
# 만약 누락된 경우 / 如有缺失:
# !pip install scikit-image pandas

from skimage.metrics import structural_similarity as ssim
import pandas as pd

print("Evaluation libraries loaded. / 평가 라이브러리 로드 완료. / 评估库已加载。")


# ---------------------------------------------------------------------------
# 1. Laplacian Variance (선명도 / 清晰度)
# ---------------------------------------------------------------------------

def laplacian_variance(img):
    """Calculate image sharpness using Laplacian variance.
    Laplacian 분산값으로 이미지 선명도를 계산합니다.
    使用Laplacian方差计算图像清晰度。

    Args:
        img: BGR 또는 grayscale 이미지 (numpy array) / BGR或灰度图像
    Returns:
        float: Laplacian variance (값이 높을수록 선명) / 值越高越清晰
    Uses: cv2.Laplacian()
    """
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    return float(lap.var())


# ---------------------------------------------------------------------------
# 2. SSIM (구조적 유사도 / 结构相似度)
# ---------------------------------------------------------------------------

def compute_ssim(img_a, img_b):
    """Compute SSIM between two images. Resizes img_b if sizes differ.
    두 이미지 간 SSIM을 계산합니다. 크기가 다르면 img_b를 img_a 크기로 조정합니다.
    计算两幅图像之间的SSIM。如果尺寸不同，将img_b调整为img_a的尺寸。

    Args:
        img_a: Reference image (BGR or grayscale) / 기준 이미지 / 参考图像
        img_b: Comparison image (BGR or grayscale) / 비교 이미지 / 比较图像
    Returns:
        float: SSIM value (0~1, higher = more similar)
    Uses: skimage.metrics.structural_similarity
    """
    if len(img_a.shape) == 3:
        a_gray = cv2.cvtColor(img_a, cv2.COLOR_BGR2GRAY)
    else:
        a_gray = img_a

    if len(img_b.shape) == 3:
        b_gray = cv2.cvtColor(img_b, cv2.COLOR_BGR2GRAY)
    else:
        b_gray = img_b

    if a_gray.shape != b_gray.shape:
        b_gray = cv2.resize(b_gray, (a_gray.shape[1], a_gray.shape[0]))

    return float(ssim(a_gray, b_gray, data_range=b_gray.max() - b_gray.min()))


# ---------------------------------------------------------------------------
# 3. 단일 이미지 평가 / 单张图像评估
# ---------------------------------------------------------------------------

def evaluate_result(pipeline_result, image_name="image"):
    """Evaluate a single pipeline result with quantitative metrics.
    단일 파이프라인 결과를 정량적 지표로 평가합니다.
    使用定量指标评估单个流水线结果。

    Args:
        pipeline_result: run_pipeline() 반환 dict / run_pipeline()返回的字典
        image_name: 이미지 이름 (표시용) / 图像名称(用于显示)
    Returns:
        dict: {image_name, success, processing_time, laplacian_before, laplacian_after,
               laplacian_ratio, ssim_value}
    """
    success = pipeline_result["success"]
    proc_time = pipeline_result["processing_time"]
    original = pipeline_result["original"]
    result = pipeline_result["result"]

    lap_before = laplacian_variance(original)
    lap_after = laplacian_variance(result)
    lap_ratio = lap_after / lap_before if lap_before > 0 else 0.0
    ssim_value = compute_ssim(original, result)

    status_kr = "성공" if success else "실패"
    status_cn = "成功" if success else "失败"
    print(f'{'='*55}')
    print(f"  Image / 이미지 / 图像: {image_name}")
    print(f"  Detection / 문서 검출 / 文档检测: {status_kr} ({status_cn})")
    print(f"  Processing time / 처리 시간 / 处理时间: {proc_time:.3f}s")
    print(f"  Laplacian Variance (Before / 전 / 前): {lap_before:.2f}")
    print(f"  Laplacian Variance (After  / 후 / 后): {lap_after:.2f}")
    print(f"  Laplacian Ratio (향상 배율 / 增强倍率): {lap_ratio:.2f}x")
    print(f"  SSIM (Original vs Result / 원본 vs 결과 / 原图vs结果): {ssim_value:.4f}")
    print(f'{'='*55}')

    return {
        "image_name": image_name,
        "success": success,
        "processing_time": proc_time,
        "laplacian_before": lap_before,
        "laplacian_after": lap_after,
        "laplacian_ratio": lap_ratio,
        "ssim_value": ssim_value,
    }


# ---------------------------------------------------------------------------
# 4. 다중 이미지 평가 요약 / 多张图像评估汇总
# ---------------------------------------------------------------------------

def evaluate_multiple(results_list):
    """Summarize evaluation results from multiple images using pandas DataFrame.
    여러 이미지의 평가 결과를 pandas DataFrame으로 요약합니다.
    使用pandas DataFrame汇总多张图像的评估结果。

    Args:
        results_list: list of dicts from evaluate_result()
    Returns:
        pd.DataFrame: summary table / 요약 테이블 / 汇总表
    """
    df = pd.DataFrame(results_list)

    df_display = df.rename(columns={
        "image_name": "Image / 이미지 / 图像",
        "success": "Detected / 검출 / 检测",
        "processing_time": "Time (s) / 시간 / 时间",
        "laplacian_before": "Laplacian Before / 전 / 处理前",
        "laplacian_after": "Laplacian After / 후 / 处理后",
        "laplacian_ratio": "Laplacian Ratio / 향상배율 / 增强倍率",
        "ssim_value": "SSIM / 유사도 / 相似度",
    })

    print("\n" + "="*80)
    print("  EVALUATION SUMMARY / 평가 요약 / 评估汇总")
    print("="*80)
    print(df_display.to_string(index=False))

    n = len(df)
    success_rate = df["success"].sum() / n * 100
    avg_time = df["processing_time"].mean()
    avg_lap_ratio = df["laplacian_ratio"].mean()
    avg_ssim = df["ssim_value"].mean()

    print(f"\n  Statistics / 통계 / 统计 (n={n}):")
    print(f"    Detection success rate / 검출 성공률 / 检测成功率: {success_rate:.1f}%")
    print(f"    Average processing time / 평균 처리 시간 / 平均处理时间: {avg_time:.3f}s")
    print(f"    Average Laplacian ratio / 평균 Laplacian 향상 배율 / 平均增强倍率: {avg_lap_ratio:.2f}x")
    print(f"    Average SSIM / 평균 SSIM / 平均SSIM: {avg_ssim:.4f}")
    print("="*80)

    return df


# ---------------------------------------------------------------------------
# 5. 비교 시각화 (평가 지표 포함) / 对比可视化(含评估指标)
# ---------------------------------------------------------------------------

def show_comparison(pipeline_result, image_name="image"):
    """Display original, perspective-corrected, and final result side-by-side with scores.
    원본, 투시 보정, 최종 결과를 점수와 함께 나란히 표시합니다.
    并排显示原图、透视矫正图和最终结果图以及评分。

    Args:
        pipeline_result: run_pipeline() 반환 dict / run_pipeline()返回的字典
        image_name: 이미지 이름 (표시용) / 图像名称(用于显示)
    Uses: show_images()
    """
    original = pipeline_result["original"]
    result = pipeline_result["result"]
    success = pipeline_result["success"]
    warped = pipeline_result.get("warped")

    lap_orig = laplacian_variance(original)
    lap_result = laplacian_variance(result)
    ssim_val = compute_ssim(original, result)

    images = []
    titles = []

    images.append(original)
    titles.append(f"Original / 원본 / 原图 [{image_name}]\nLaplacian: {lap_orig:.1f}")

    if warped is not None and success:
        lap_warped = laplacian_variance(warped)
        images.append(warped)
        titles.append(f"Perspective Corrected / 투시 보정 / 透视矫正\nLaplacian: {lap_warped:.1f}")

    images.append(result)
    titles.append(f"Enhanced (Final) / 향상 결과 / 增强结果\nLaplacian: {lap_result:.1f}\nSSIM: {ssim_val:.4f}")

    show_images(images, titles, cols=len(images), figsize=(6 * len(images), 5))


print("Evaluation functions defined. / 평가 함수 정의 완료. / 评估函数已定义。")
print("-" * 55)
print("Usage example / 사용 예시 / 使用示例:")
print("  res = run_pipeline('doc.jpg')")
print("  metrics = evaluate_result(res, image_name='test1')")
print("  show_comparison(res, image_name='test1')")
print()
print("  # Multiple images summary / 여러 이미지 요약 / 多图汇总:")
print("  all_metrics = [evaluate_result(r, n) for r, n in zip(results, names)]")
print("  df = evaluate_multiple(all_metrics)")


---
## 8. Upload Document Images / 문서 이미지 업로드 / 上传文档图像

아래 셀을 실행하여 이미지를 업로드하고 처리하세요.
运行下方单元格上传图像并进行处理。

### 옵션 A / 选项A: 직접 업로드 (권장 / 推荐)
아래 버튼을 클릭하여 이미지 파일을 선택하세요. 여러 장 선택 가능합니다.
点击下方按钮选择图像文件，可多选。

In [ ]:
# 8-A. 이미지 업로드 / 图像上传 (지원 형식 / 支持格式: JPG, PNG, BMP)

print("이미지 파일을 선택하세요 (여러 장 가능)... / 请选择图像文件（可多选）...")
uploaded = files.upload()

input_images = {}
for fname, data in uploaded.items():
    if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
        # Save uploaded file / 업로드된 파일 저장 / 保存上传文件
        with open(fname, 'wb') as f:
            f.write(data)
        img = cv2.imread(fname)
        if img is not None:
            input_images[fname] = img
            print(f"  [OK] {fname} — {img.shape[1]}x{img.shape[0]}")
        else:
            print(f"  [FAIL/실패/失败] {fname} — could not read as image / 이미지 읽기 실패 / 无法读取为图像")
    else:
        print(f"  [SKIP/건너뜀/跳过] {fname} — unsupported format / 지원하지 않는 형식 / 不支持的格式")

if not input_images:
    print("\n⚠️ No valid images uploaded. Please upload JPG, PNG, or BMP files.")
    print("   유효한 이미지가 없습니다. JPG, PNG, BMP 파일을 업로드하세요.")
    print("   没有有效图像，请上传 JPG、PNG 或 BMP 文件。")
else:
    print(f"\n✅ {len(input_images)} image(s) ready for processing. / 처리 준비 완료. / 图像已准备好处理。")

### 옵션 B / 选项B: Google Drive에서 이미지 가져오기 / 从Google Drive获取图像
Google Drive를 마운트하고 이미지 경로를 지정하세요.
挂载Google Drive并指定图像路径。

In [ ]:
# 8-B. (선택 / 可选) Google Drive 마운트 / 挂载Google Drive
# 필요한 경우에만 아래 코드를 실행하세요.
# 仅在需要时运行以下代码。

use_drive = False  # True로 변경하여 Google Drive 사용 / 改为True以使用Google Drive

if use_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # 이미지가 있는 폴더 경로 (자신의 경로로 변경하세요)
    # 图像所在文件夹路径（请改为自己的路径）
    DRIVE_IMAGE_DIR = "/content/drive/MyDrive/document_images"
    
    if os.path.isdir(DRIVE_IMAGE_DIR):
        input_images = {}
        for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp'):
            for fpath in glob.glob(os.path.join(DRIVE_IMAGE_DIR, ext)):
                fpath_lower = fpath.lower()
                if fpath_lower.endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                    img = cv2.imread(fpath)
                    if img is not None:
                        fname = os.path.basename(fpath)
                        input_images[fname] = img
                        print(f"  [OK] {fname}")
        print(f"\n✅ {len(input_images)} image(s) loaded from Drive. / Drive에서 로드 완료. / 从Drive加载完成。")
    else:
        print(f"⚠️ Directory not found / 디렉터리 없음 / 目录不存在: {DRIVE_IMAGE_DIR}")
        print("Please update DRIVE_IMAGE_DIR to your actual image folder path.")
        print("DRIVE_IMAGE_DIR를 실제 이미지 폴더 경로로 변경하세요.")
        print("请将DRIVE_IMAGE_DIR更新为您实际的图像文件夹路径。")

---
## 9. Run Pipeline / 파이프라인 실행 / 运行流水线

업로드한 모든 이미지를 처리하고 결과를 출력합니다.
处理所有上传的图像并输出结果。

In [ ]:
# 9. 업로드된 모든 이미지에 대해 파이프라인 실행
# 9. 对所有上传的图像运行流水线

if 'input_images' not in dir() or not input_images:
    print("⚠️ 먼저 이미지를 업로드하세요! (옵션 A 또는 B 실행)")
    print("   请先上传图像！（运行选项A或B）")
else:
    OUTPUT_DIR = "output"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    success_count = 0
    total_time = 0.0
    all_results = []
    
    for fname, img in input_images.items():
        basename = os.path.splitext(fname)[0]
        print(f"{'='*60}")
        print(f"[INFO] Processing / 처리 중 / 处理中: {fname} ({img.shape[1]}x{img.shape[0]})")
        
        result = run_pipeline(img, OUTPUT_DIR, save_steps=True, basename=basename)
        
        print(f"[INFO] Document detected / 문서 검출 / 文档检测: {result['success']}")
        print(f"[INFO] Processing time / 처리 시간 / 处理时间: {result['processing_time']:.3f}s")
        print(f"[INFO] Saved / 저장됨 / 已保存: {result['output_path']}")
        
        if result['success']:
            success_count += 1
        total_time += result['processing_time']
        all_results.append((basename, result))
    
    total = len(input_images)
    print(f"\n{'='*60}")
    print(f"[SUMMARY/요약/总结] Detection success / 검출 성공 / 检测成功: {success_count}/{total} ({100*success_count/total:.1f}%)")
    print(f"[SUMMARY/요약/总结] Total processing time / 총 처리 시간 / 总处理时间: {total_time:.3f}s")
    print(f"[SUMMARY/요약/总结] Average time / 평균 시간 / 平均时间: {total_time/total:.3f}s/image")

---
## 10. Visualize Results / 결과 시각화 / 结果可视化

각 이미지의 단계별 중간 결과를 확인합니다.
查看每张图像的各步骤中间结果。

In [ ]:
# 10. 결과 시각화 / 结果可视化

if 'all_results' not in dir() or not all_results:
    print("⚠️ 먼저 파이프라인을 실행하세요! / 请先运行流水线！")
else:
    for basename, result in all_results:
        print(f"\n{'='*60}")
        status_emoji = "✅" if result['success'] else "⚠️"
        print(f"{status_emoji} {basename} — {result['processing_time']:.3f}s")
        
        # Original vs Result comparison / 원본 vs 결과 비교 / 原图vs结果对比
        original = result['original']
        final = result['result']
        warped = result.get('warped')
        
        images = [original]
        titles = ["01. Original / 원본 / 原图"]
        
        if warped is not None and result['success']:
            images.append(warped)
            titles.append("06. Perspective Corrected / 투시 보정 / 透视矫正")
        
        images.append(final)
        titles.append("07. Enhanced (Final) / 향상 결과 / 增强结果")
        
        show_images(images, titles, cols=len(images), figsize=(6*len(images), 5))
        
        # Show all 7 steps if available / 7단계 전체 표시 / 显示全部7步
        steps = result.get('steps_images', {})
        if steps:
            step_names = ['01_original', '02_grayscale', '03_blurred', 
                         '04_edges', '05_contours', '06_perspective', '07_enhanced']
            step_titles_kr = ['원본', '그레이스케일', '가우시안 블러', 
                            '에지 검출', '윤곽선 검출', '투시 변환', '향상 결과']
            step_titles_cn = ['原图', '灰度化', '高斯模糊', 
                            '边缘检测', '轮廓检测', '透视变换', '增强结果']
            step_images = [steps[name] for name in step_names if name in steps]
            step_titles = [f"{name.replace('_',' ').title()}\n{kr}\n{cn}" 
                          for name, kr, cn in zip(step_names, step_titles_kr, step_titles_cn) 
                          if name in steps]
            if step_images:
                print(f"\n  --- All 7 Steps for {basename} / 전체 7단계 / 全部7步骤 ---")
                show_images(step_images, step_titles, cols=4, figsize=(16, 10))

---
## 11. Download Results / 결과 다운로드 / 下载结果

처리된 이미지와 중간 결과를 ZIP 파일로 다운로드합니다.
将处理后的图像和中间结果下载为ZIP文件。

In [ ]:
# 11. 결과 다운로드 / 下载结果

import shutil

if os.path.isdir(OUTPUT_DIR if 'OUTPUT_DIR' in dir() else 'output'):
    output_zip = 'document_correction_results.zip'
    shutil.make_archive('document_correction_results', 'zip', 
                        OUTPUT_DIR if 'OUTPUT_DIR' in dir() else 'output')
    files.download(output_zip)
    print(f"✅ Results downloaded / 결과 다운로드 완료 / 结果已下载: {output_zip}")
else:
    print("⚠️ No output directory found. Run the pipeline first.")
    print("   출력 디렉터리가 없습니다. 파이프라인을 먼저 실행하세요.")
    print("   未找到输出目录，请先运行流水线。")

---
## 12. 사용 방법 요약 / 使用说明 / Usage Summary

### 실행 순서 / 执行顺序

| 순서 | 내용 | 内容 |
|------|------|------|
| 1 | 환경 설정 / 环境设置 | 상단 코드 셀 순서대로 실행 / 按顺序运行上方代码单元格 (`Runtime → Run all`) |
| 2 | 이미지 업로드 / 上传图像 | 셀 8-A 실행 후 이미지 파일 선택 (JPG/PNG/BMP) / 运行单元格8-A后选择图像文件 |
| 3 | 처리 실행 / 执行处理 | 셀 9에서 모든 이미지 자동 처리 / 单元格9自动处理所有图像 |
| 4 | 결과 확인 / 查看结果 | 셀 10에서 단계별 결과 시각화 / 单元格10中查看各步骤结果可视化 |
| 5 | 다운로드 / 下载 | 셀 11에서 ZIP 파일 다운로드 / 单元格11下载ZIP文件 |

### 처리 과정 / 处理步骤

| Step | 한국어 | 中文 | English |
|------|--------|------|---------|
| 01 | 원본 이미지 | 原始图像 | Original Image |
| 02 | 그레이스케일 변환 | 灰度转换 | Grayscale Conversion |
| 03 | 가우시안 블러 | 高斯模糊 | Gaussian Blur |
| 04 | 케니 에지 검출 | Canny边缘检测 | Canny Edge Detection |
| 05 | 윤곽선 검출 | 轮廓检测 | Contour Detection |
| 06 | 투시 변환 | 透视变换 | Perspective Transform |
| 07 | 향상 결과 | 增强结果 | Enhanced Result |

### 파라미터 조정 / 参数调整

검출 성능을 개선하려면 아래 파라미터를 조정해보세요.
要改善检测性能，请尝试调整以下参数。

- Canny thresholds: `detect_edges(blurred, low=50, high=150)`
- Gaussian kernel: `apply_gaussian_blur(gray, kernel_size=(7,7))`
- CLAHE clip limit: `apply_clahe(gray, clip_limit=3.0)`